# Phase 1 -- Wind Prediction Starting Kit

This notebook implements a simple of **CatBoost quantile** for each horizon. 

**Prerequisites**: Run `1_feature_engineering.ipynb` first to generate pre-computed features in `phase1_dataset/features/`.

**Runtime**: ~5-30 minutes.



## 1. Setup

In [1]:
import os, sys
# Ensure we're in the notebook's directory (participant_kit/phase1/)
_this_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else None
if _this_dir is None:
    for _candidate in [os.getcwd(), os.path.join(os.getcwd(), "participant_kit", "phase1")]:
        if os.path.exists(os.path.join(_candidate, "utils.py")):
            os.chdir(_candidate)
            break
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import warnings
import time as _time

import numpy as np
import pandas as pd
import lightgbm as lgb
from pathlib import Path
from catboost import CatBoostRegressor

from utils import (
    REGIONS, HORIZONS, HOURS, QUANTILES_DEFAULT,
    load_train_data, load_vertical_ratios,
    exclude_worldwide_features, load_or_compute_selection,
    winkler_score, circular_mae,
    predict_speed_10m, predict_direction, merge_speed_direction,
    generate_submission,
)

warnings.filterwarnings("ignore")

DATA_DIR = Path("phase1_dataset")
FEATURES_DIR = DATA_DIR / "features"
RECOMPUTE_SELECTION = False  # Set True to recompute feature importance (~2 min)

assert FEATURES_DIR.exists(), (
    f"Features directory not found: {FEATURES_DIR}\n"
    "Run 1_feature_engineering.ipynb first."
)
print("Setup OK")


Setup OK


## 2. Load Data

Load pre-computed training features for both regions. Each region has ~2.8M rows
(3 years x 365 days x ~2565 grid points) with ~130 features including reanalysis variables,
HRES NWP forecasts, lags, and rolling statistics.


In [2]:
# Load training data for both regions
data = {}  # {region: (df, feature_cols, speed_targets, dir_targets)}

for region in REGIONS:
    df, feature_cols, speed_targets, dir_targets = load_train_data(FEATURES_DIR, region)
    data[region] = (df, feature_cols, speed_targets, dir_targets)
    print(f"\n{region}:")
    print(f"  Shape: {df.shape}")
    print(f"  Time range: {df['time'].min().date()} to {df['time'].max().date()}")
    print(f"  Features: {len(feature_cols)}, Speed targets: {len(speed_targets)}, Dir targets: {len(dir_targets)}")
    print(f"  Grid points: {df.groupby(['latitude', 'longitude']).ngroups}")


north_sea:
  Shape: (2811240, 270)
  Time range: 2019-01-01 to 2021-12-31
  Features: 245, Speed targets: 12, Dir targets: 12
  Grid points: 2565

east_china_sea:
  Shape: (2811240, 270)
  Time range: 2019-01-01 to 2021-12-31
  Features: 245, Speed targets: 12, Dir targets: 12
  Grid points: 2565


## 3. Feature Selection (skippable)

Select the most important features per target using a quick CatBoost importance ranking.
Results are cached to JSON -- set `RECOMPUTE_SELECTION = True` above to rerun.

Short-term targets (+1d) need fewer features (dominated by HRES forecasts),
while long-term targets (+14d) benefit from more reanalysis context.


In [3]:
selected_features = {}  # {region: {target: [feature_names]}}

for region in REGIONS:
    df, feature_cols, speed_targets, dir_targets = data[region]
    cache_path = FEATURES_DIR / f"selected_features_catboost_{region}.json"
    selected_features[region] = load_or_compute_selection(
        cache_path, df, feature_cols, speed_targets,
        model_type="catboost",
        top_k={1: 15, 7: 20, 14: 25},
        force=RECOMPUTE_SELECTION,
    )

# Show a sample
sample_region = REGIONS[0]
sample_target = data[sample_region][2][0]  # first speed target
print(f"\nExample: {sample_region} / {sample_target}")
print(f"  Selected ({len(selected_features[sample_region][sample_target])} feats): "
      f"{selected_features[sample_region][sample_target]}")

Computing feature selection (catboost)...
Saved: selected_features_catboost_north_sea.json
Computing feature selection (catboost)...
Saved: selected_features_catboost_east_china_sea.json

Example: north_sea / speed_d14_h0
  Selected (25 feats): ['sst', 'woy_cos', 'elevation', 'woy_sin', 'latitude', 'up_north_atlantic_msl_lag5d', 'up_iceland_msl_lag1d', 'up_greenland_tip_ws10_lag3d', 'up_iceland_ws10_lag1d', 'up_iberian_msl_lag3d', 'ecs_pressure_gradient', 'up_mid_atlantic_ws10', 'up_iceland_msl_lag5d', 'up_iceland_ws10_lag5d', 'up_iberian_msl_lag1d', 'up_iberian_msl_lag5d', 'natl_pc6', 'natl_pc4', 'siberian_high', 'ws10_rmean7d', 'up_greenland_tip_msl_lag1d', 'ns_pressure_gradient', 'up_greenland_tip_msl', 'fcst_speed_d7_h6', 'up_mid_atlantic_msl_lag5d']


## 4. Train/Val Split

- **Train**: 2019--2020
- **Validation**: full 2021 (held out for early stopping and evaluation)
- **Subsample**: 500K rows max per region to keep training fast
- **Speed models**: exclude worldwide features (NAO, Siberian High, etc.) -- local reanalysis + HRES suffice
- **Direction models**: keep all features and let per-target selection pick the best

In [4]:
MAX_TRAIN_SAMPLES = 500_000

splits = {}  # {region: (df_train, df_val, feature_cols_speed, feature_cols_dir)}

for region in REGIONS:
    df, feature_cols, speed_targets, dir_targets = data[region]

    # Train: 2019-2020, Val: full 2021
    train_mask = df["time"].dt.year.isin([2019, 2020])
    val_mask = df["time"].dt.year == 2021

    df_train = df[train_mask].copy()
    df_val = df[val_mask].copy()

    # Subsample training data
    if len(df_train) > MAX_TRAIN_SAMPLES:
        df_train = df_train.sample(n=MAX_TRAIN_SAMPLES, random_state=42)

    # Speed models: exclude worldwide features
    feature_cols_speed = exclude_worldwide_features(feature_cols)
    # Direction models: keep all features (selection will pick the best)
    feature_cols_dir = feature_cols

    splits[region] = (df_train, df_val, feature_cols_speed, feature_cols_dir)
    print(f"{region}: train={len(df_train):,}, val={len(df_val):,}, "
          f"speed_feats={len(feature_cols_speed)}, dir_feats={len(feature_cols_dir)}")

north_sea: train=500,000, val=936,225, speed_feats=194, dir_feats=245
east_china_sea: train=500,000, val=936,225, speed_feats=194, dir_feats=245


## 5. Speed Models -- CatBoost Quantile Regression

Train per-target CatBoost models for three quantiles (q05, q50, q95).
Uses the feature selection from step 3 and early stopping on validation data.

This trains 12 targets x 3 quantiles x 2 regions = 72 models.

In [5]:
QUANTILES = QUANTILES_DEFAULT  # [0.05, 0.5, 0.95]

speed_models = {}  # {region: {target: {quantile: model}}}

t0 = _time.time()

for region in REGIONS:
    df_train, df_val, feature_cols_speed, _ = splits[region]
    _, _, speed_targets, _ = data[region]
    sel = selected_features[region]

    speed_models[region] = {}
    print(f"\n{'='*60}")
    print(f"Training speed models: {region} ({len(speed_targets)} targets x {len(QUANTILES)} quantiles)")
    print(f"{'='*60}")

    for tgt in speed_targets:
        feats = sel.get(tgt, feature_cols_speed)

        # Prepare data, drop rows where target is NaN
        mask_tr = df_train[tgt].notna()
        mask_vl = df_val[tgt].notna()
        X_tr = df_train.loc[mask_tr, feats].fillna(0)
        y_tr = df_train.loc[mask_tr, tgt]
        X_vl = df_val.loc[mask_vl, feats].fillna(0)
        y_vl = df_val.loc[mask_vl, tgt]

        models_q = {}
        for q in QUANTILES:
            m = CatBoostRegressor(
                loss_function=f"Quantile:alpha={q}",
                iterations=500, depth=7, learning_rate=0.05,
                l2_leaf_reg=3, random_seed=42, verbose=0,
            )
            m.fit(X_tr, y_tr, eval_set=(X_vl, y_vl), early_stopping_rounds=50)
            models_q[q] = m
        speed_models[region][tgt] = models_q

        # Quick validation metrics
        q05 = np.maximum(models_q[0.05].predict(X_vl), 0)
        q50 = models_q[0.5].predict(X_vl)
        q95 = models_q[0.95].predict(X_vl)
        rmse = float(np.sqrt(np.nanmean((y_vl - q50) ** 2)))
        ws = winkler_score(y_vl.values, q05, q95, alpha=0.1)
        print(f"  {tgt}: RMSE={rmse:.3f}, WS={ws:.3f}")

elapsed = _time.time() - t0
print(f"\nSpeed training complete in {elapsed:.0f}s")


Training speed models: north_sea (12 targets x 3 quantiles)
  speed_d14_h0: RMSE=2.914, WS=11.089
  speed_d14_h12: RMSE=2.995, WS=11.627
  speed_d14_h18: RMSE=2.963, WS=11.351
  speed_d14_h6: RMSE=2.940, WS=11.277
  speed_d1_h0: RMSE=0.931, WS=3.807
  speed_d1_h12: RMSE=1.137, WS=4.752
  speed_d1_h18: RMSE=1.213, WS=5.133
  speed_d1_h6: RMSE=1.019, WS=4.222
  speed_d7_h0: RMSE=2.792, WS=10.822
  speed_d7_h12: RMSE=2.888, WS=11.268
  speed_d7_h18: RMSE=2.865, WS=10.990
  speed_d7_h6: RMSE=2.834, WS=11.027

Training speed models: east_china_sea (12 targets x 3 quantiles)
  speed_d14_h0: RMSE=2.776, WS=11.074
  speed_d14_h12: RMSE=2.821, WS=11.124
  speed_d14_h18: RMSE=2.795, WS=11.072
  speed_d14_h6: RMSE=2.892, WS=11.703
  speed_d1_h0: RMSE=1.008, WS=4.071
  speed_d1_h12: RMSE=1.197, WS=4.896
  speed_d1_h18: RMSE=1.259, WS=5.182
  speed_d1_h6: RMSE=1.110, WS=4.620
  speed_d7_h0: RMSE=2.361, WS=9.456
  speed_d7_h12: RMSE=2.452, WS=10.011
  speed_d7_h18: RMSE=2.470, WS=9.973
  speed_d7_h

## 6. Direction Models -- LightGBM sin/cos

Wind direction is circular (0 = 360 degrees). We decompose into sin/cos components,
train LightGBM regressors on each, and reconstruct direction via `atan2`.

Feature selection: top-25 features per target (computed inline via quick LGBM importance).

In [6]:
# Direction Models — LGBM sin/cos
# Self-contained: loads data directly, no dependency on other cells' variables
import lightgbm as lgb
from utils import load_train_data, exclude_worldwide_features, circular_mae
import time as _time

dir_models = {}
t0 = _time.time()

for region in REGIONS:
    df_dir, feature_cols_dir, _, dir_targets_list = load_train_data(FEATURES_DIR, region)
    df_train_dir = df_dir[df_dir["time"].dt.year.isin([2019, 2020])]
    
    dir_models[region] = {}
    for tgt in dir_targets_list:
        mask_tr = df_train_dir[tgt].notna()
        if mask_tr.sum() < 100:
            continue
        
        # Feature selection (top-25)
        X_sub = df_train_dir.loc[mask_tr, feature_cols_dir].fillna(0).sample(
            n=min(100_000, mask_tr.sum()), random_state=42)
        y_sub = df_train_dir.loc[X_sub.index, tgt]
        m_imp = lgb.LGBMRegressor(n_estimators=50, max_depth=4, verbose=-1, n_jobs=-1)
        m_imp.fit(X_sub, np.sin(np.radians(y_sub)))
        dir_feats = pd.Series(m_imp.feature_importances_, index=feature_cols_dir).nlargest(25).index.tolist()
        
        X_tr = df_train_dir.loc[mask_tr, dir_feats].fillna(0)
        y_tr_sin = np.sin(np.radians(df_train_dir.loc[mask_tr, tgt]))
        y_tr_cos = np.cos(np.radians(df_train_dir.loc[mask_tr, tgt]))
        
        params = dict(n_estimators=200, max_depth=7, learning_rate=0.05,
                      subsample=0.8, verbose=-1, n_jobs=-1)
        m_sin = lgb.LGBMRegressor(**params)
        m_cos = lgb.LGBMRegressor(**params)
        m_sin.fit(X_tr, y_tr_sin)
        m_cos.fit(X_tr, y_tr_cos)
        dir_models[region][tgt] = (m_sin, m_cos, dir_feats)
    
    print(f"  {region}: {len(dir_models[region])} direction models trained")
    del df_dir, df_train_dir

print(f"Direction training complete in {_time.time() - t0:.0f}s")


  north_sea: 12 direction models trained
  east_china_sea: 12 direction models trained
Direction training complete in 535s


In [7]:
dir_offsets = {}
try:
    # Compute direction prediction intervals
    from utils import compute_direction_intervals
    
    dir_offsets = {}
    for region in REGIONS:
        df_tr_dir, fcols_dir, _, dir_tgts = load_train_data(FEATURES_DIR, region)
        df_tr_dir = df_tr_dir[df_tr_dir["time"].dt.year.isin([2019, 2020])]
        dir_offsets[region] = compute_direction_intervals(
            df_tr_dir, dir_tgts, fcols_dir, dir_models[region], quantile_hi=0.975)
        print(f"  {region}: " + ", ".join(f"+{h}d: +/-{v:.1f} deg" for h, v in dir_offsets[region].items()))
        del df_tr_dir
    
except Exception as e:
    print(f"Dir interval computation failed: {e}")


  north_sea: +1d: +/-62.6 deg, +7d: +/-138.2 deg, +14d: +/-136.1 deg
  east_china_sea: +1d: +/-87.4 deg, +7d: +/-142.8 deg, +14d: +/-144.8 deg


## 7. Evaluation Summary

Aggregate validation metrics by region and horizon.

In [8]:
print(f"{'Region':<18} {'Horizon':<9} {'Mean WS':>8} {'Mean RMSE':>10} {'Mean cMAE':>10}")
print("-" * 58)

for region in REGIONS:
    _, df_val, _, _ = splits[region]
    _, _, speed_targets, dir_targets = data[region]
    sel = selected_features[region]

    for h in HORIZONS:
        # Speed metrics for this horizon
        ws_list, rmse_list = [], []
        for tgt in speed_targets:
            if int(tgt.split("_")[1][1:]) != h:
                continue
            feats = sel.get(tgt, [])
            mask_vl = df_val[tgt].notna()
            X_vl = df_val.loc[mask_vl, feats].fillna(0)
            y_vl = df_val.loc[mask_vl, tgt].values
            mq = speed_models[region][tgt]
            q05 = np.maximum(mq[0.05].predict(X_vl), 0)
            q50 = mq[0.5].predict(X_vl)
            q95 = mq[0.95].predict(X_vl)
            ws_list.append(winkler_score(y_vl, q05, q95, alpha=0.1))
            rmse_list.append(float(np.sqrt(np.nanmean((y_vl - q50) ** 2))))

        # Direction metrics for this horizon
        cmae_list = []
        for tgt in dir_targets:
            if int(tgt.split("_")[1][1:]) != h:
                continue
            m_sin, m_cos, dir_feats = dir_models[region][tgt]
            mask_vl = df_val[tgt].notna()
            X_vl = df_val.loc[mask_vl, dir_feats].fillna(0)
            y_vl = df_val.loc[mask_vl, tgt].values
            pred = np.degrees(np.arctan2(m_sin.predict(X_vl), m_cos.predict(X_vl))) % 360
            cmae_list.append(circular_mae(y_vl, pred))

        mean_ws = np.mean(ws_list) if ws_list else float("nan")
        mean_rmse = np.mean(rmse_list) if rmse_list else float("nan")
        mean_cmae = np.mean(cmae_list) if cmae_list else float("nan")
        print(f"{region:<18} +{h:<7d} {mean_ws:>8.3f} {mean_rmse:>10.3f} {mean_cmae:>9.1f} deg")

Region             Horizon    Mean WS  Mean RMSE  Mean cMAE
----------------------------------------------------------
north_sea          +1          4.479      1.075      13.8 deg
north_sea          +7         11.027      2.845      64.6 deg
north_sea          +14        11.336      2.953      80.3 deg
east_china_sea     +1          4.692      1.144      15.9 deg
east_china_sea     +7          9.865      2.432      44.2 deg
east_china_sea     +14        11.243      2.821      65.0 deg


## 8. Generate Submission

Load inference features for all 8 windows, predict at 10m, expand to 7 vertical levels
using climatological wind profile ratios, and save as `predictions.csv`.

In [9]:
# Prepare inputs for generate_submission()
feature_cols_all = {}
vertical_ratios = {}

for region in REGIONS:
    _, feature_cols, _, _ = data[region]
    feature_cols_all[region] = feature_cols
    vertical_ratios[region] = load_vertical_ratios(FEATURES_DIR, region)

print("Generating submission...")
sub = generate_submission(
    speed_models=speed_models,
    dir_models=dir_models,
    selected_features=selected_features,
    feature_cols_all=feature_cols_all,
    vertical_ratios=vertical_ratios,
    features_dir=FEATURES_DIR,
    regions=REGIONS,
    n_windows=8,
    quantiles=QUANTILES,
    dir_offsets=dir_offsets,
)
print(f"Submission: {len(sub):,} rows, {sub['level'].nunique()} levels")
print(f"Columns: {sub.columns.tolist()}")


Generating submission...
  W1 north_sea: grid=215,460 (7 levels), station=96
  W1 east_china_sea: grid=215,460 (7 levels), station=84
  W2 north_sea: grid=215,460 (7 levels), station=96
  W2 east_china_sea: grid=215,460 (7 levels), station=84
  W3 north_sea: grid=215,460 (7 levels), station=96
  W3 east_china_sea: grid=215,460 (7 levels), station=84
  W4 north_sea: grid=215,460 (7 levels), station=96
  W4 east_china_sea: grid=215,460 (7 levels), station=84
  W5 north_sea: grid=215,460 (7 levels), station=96
  W5 east_china_sea: grid=215,460 (7 levels), station=84
  W6 north_sea: grid=215,460 (7 levels), station=96
  W6 east_china_sea: grid=215,460 (7 levels), station=84
  W7 north_sea: grid=215,460 (7 levels), station=96
  W7 east_china_sea: grid=215,460 (7 levels), station=84
  W8 north_sea: grid=215,460 (7 levels), station=96
  W8 east_china_sea: grid=215,460 (7 levels), station=84
Submission: 3,448,800 rows, 8 levels
Columns: ['type', 'window', 'region', 'latitude', 'longitude', 'st

In [10]:
# Save submission
out_path = Path("predictions_light.csv")
sub.to_csv(out_path, index=False)
print(f"Saved: {out_path} ({len(sub):,} rows)")

# --- Package for Codabench upload ---
# Codabench expects a ZIP containing a file named exactly "predictions.csv".
import zipfile
submission_zip = Path("submission_light.zip")
with zipfile.ZipFile(submission_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(out_path, arcname="predictions.csv")
print(f"Ready to upload to Codabench: {submission_zip}")

# Breakdown
print(f"\nRows per region:")
for region in REGIONS:
    n = len(sub[sub["region"] == region])
    print(f"  {region}: {n:,}")

print(f"\nRows per level:")
for level in sub["level"].unique():
    n = len(sub[sub["level"] == level])
    print(f"  {level}: {n:,}")

print(f"\nWindows: {sub['window'].nunique()}, "
      f"Horizons: {sorted(sub['horizon'].unique())}, "
      f"Hours: {sorted(sub['hour'].unique())}")

print(f"\nSample rows:")
sub.head(5)

Saved: predictions_light.csv (3,448,800 rows)
Ready to upload to Codabench: submission_light.zip

Rows per region:
  north_sea: 1,724,448
  east_china_sea: 1,724,352

Rows per level:
  10m: 492,480
  100m: 492,480
  1000: 492,480
  925: 492,480
  850: 492,480
  700: 492,480
  500: 492,480
  : 1,440

Windows: 8, Horizons: [np.int64(1), np.int64(7), np.int64(14)], Hours: [np.int64(0), np.int64(6), np.int64(12), np.int64(18)]

Sample rows:


,type,window,region,latitude,longitude,station,horizon,hour,level,q05,q50,q95,dir_05,dir_50,dir_95
0,grid,1,north_sea,51.0,-4.00,,14,0,10m,1.5211,4.4665,9.4320,92.8,228.9,5.0
1,grid,1,north_sea,51.0,-3.75,,14,0,10m,1.5532,4.4844,9.4037,92.8,228.9,5.0
2,grid,1,north_sea,51.0,-3.50,,14,0,10m,1.5180,4.5418,9.3649,93.8,229.9,6.0
3,grid,1,north_sea,51.0,-3.25,,14,0,10m,1.4456,4.5797,9.2393,94.2,230.3,6.4
4,grid,1,north_sea,51.0,-3.00,,14,0,10m,1.4456,4.5797,9.2393,94.2,230.3,6.4
